<a href="https://colab.research.google.com/github/tensorbytes0202/Deep-learning/blob/main/Bert_sentiment_and_Bart_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install transformers datasets

In [9]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import TrainingArguments, Trainer
import torch

In [10]:
dataset = load_dataset("imdb")
# 🔥 small subset for fast training

train_data = dataset["train"].shuffle(seed=42).select(range(1500))
test_data = dataset["test"].shuffle(seed=42).select(range(500))

In [11]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [12]:
def tokenize(example):
    return tokenizer(example['text'], padding='max_length', truncation=True, max_length=128)

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data = train_data.rename_column("label", "labels")
test_data = test_data.rename_column("label", "labels")

train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [13]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
training_args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    logging_steps=20,
    eval_strategy="epoch",
    fp16=True
)

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.273242,0.378066
2,0.254765,0.415932
3,0.098830,0.753702
4,0.049916,0.809115
5,0.002452,0.817979


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=470, training_loss=0.1478602293958055, metrics={'train_runtime': 79.4783, 'train_samples_per_second': 94.365, 'train_steps_per_second': 5.914, 'total_flos': 493333228800000.0, 'train_loss': 0.1478602293958055, 'epoch': 5.0})

In [21]:
trainer.evaluate()

{'eval_loss': 0.8179788589477539,
 'eval_runtime': 1.0478,
 'eval_samples_per_second': 477.196,
 'eval_steps_per_second': 30.541,
 'epoch': 5.0}

In [22]:
text = "This movie was absolutely amazing and fantastic!"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to("cuda")

model.to("cuda")
outputs = model(**inputs)

pred = torch.argmax(outputs.logits)

print("Positive" if pred.item() == 1 else "Negative")

Positive


BART TEXT SUMMARIZATION

In [23]:
from datasets import load_dataset
from transformers import BartTokenizer, BartForConditionalGeneration
from transformers import TrainingArguments, Trainer

In [24]:
dataset = load_dataset("xsum")

# 🔥 small subset
train_data = dataset["train"].select(range(500))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

In [25]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')
model = BartForConditionalGeneration.from_pretrained('facebook/bart-base')

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [26]:
def preprocess(example):
    inputs = tokenizer(example['document'], max_length=256, truncation=True, padding='max_length')
    outputs = tokenizer(example['summary'], max_length=64, truncation=True, padding='max_length')

    inputs['labels'] = outputs['input_ids']
    return inputs

train_data = train_data.map(preprocess, batched=True)

train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [28]:
training_args = TrainingArguments(
    output_dir='./bart_results',
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=20,
    fp16=True
)

In [29]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_data,

)

In [30]:
trainer.train()

Step,Training Loss
20,8.138186
40,4.464652
60,3.147079
80,2.415871
100,2.136415
120,1.761471


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=125, training_loss=3.6022052459716796, metrics={'train_runtime': 21.0675, 'train_samples_per_second': 23.733, 'train_steps_per_second': 5.933, 'total_flos': 76217057280000.0, 'train_loss': 3.6022052459716796, 'epoch': 1.0})

In [31]:
text = """
Artificial Intelligence is transforming industries by automating tasks and improving efficiency.
It is widely used in healthcare, finance, and education.
"""

inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True).to("cuda")

model.to("cuda")
summary_ids = model.generate(inputs['input_ids'], max_length=50)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Summary:", summary)

Summary: Artificial Intelligence (Artificial intelligence) has the power to improve efficiency and efficiency.
